# Notebook 4: Evaluator-Optimiser LLM Pattern

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain the **Evaluator-Optimiser** pattern and its self-correcting feedback loop.
2. Understand why separating **generation** from **evaluation** improves output quality.
3. Use **Pydantic** with `with_structured_output()` to enforce strict JSON responses from LLMs.
4. Build a feedback loop that iteratively improves outputs until they meet all criteria.
5. Run a working **Investment Memo Generator** with built-in compliance checking.

---

## Use Case: Compliant Investment Memo Generator

> **Scenario**: A financial services firm needs to generate investment memos that comply with regulatory standards. Every memo must include specific elements (investment recommendation, key metrics, risk disclaimer). A single LLM call often misses one or more requirements. The Evaluator-Optimiser pattern ensures compliance automatically.

---

## The Evaluator-Optimiser Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                       REQUEST                               │
│         "Write a memo about TechCorp Q3 earnings"          │
└──────────────────────────┬──────────────────────────────────┘
                           │
                           ▼  Attempt 1
┌─────────────────────────────────────────────────────────────┐
│                    GENERATOR LLM                            │
│                  (temperature=0.7)                          │
│                                                             │
│  Writes initial draft memo...                               │
└──────────────────────────┬──────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────┐
│                    EVALUATOR LLM                            │
│                  (temperature=0.0)                          │
│                                                             │
│  Reviews against criteria:                                  │
│  - Has Investment Recommendation?                           │
│  - Missing Revenue/EPS figures                              │
│  - Missing risk disclaimer                                  │
│                                                             │
│  → decision: "FAIL", score: 6, feedback: "Add metrics..."  │
└──────────────────────────┬──────────────────────────────────┘
                           │ FAIL → loop back
                           ▼  Attempt 2
┌─────────────────────────────────────────────────────────────┐
│                    GENERATOR LLM                            │
│                                                             │
│  Revises draft using feedback...                            │
└──────────────────────────┬──────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────┐
│                    EVALUATOR LLM                            │
│                                                             │
│  - Has Investment Recommendation                            │
│  - Has Revenue and EPS                                      │
│  - Has risk disclaimer                                      │
│                                                             │
│  → decision: "PASS", score: 9                               │
└──────────────────────────┬──────────────────────────────────┘
                           │ PASS → done
                           ▼
                    FINAL APPROVED MEMO
```

---

## Key Concepts

| Concept | Description |
|---|---|
| **Generator** | Creative LLM (higher temperature) that writes the initial draft and revisions |
| **Evaluator** | Strict LLM (temperature=0) that grades the draft against fixed criteria |
| **Structured Output** | `with_structured_output(Pydantic)` forces the LLM to return valid JSON |
| **Feedback Loop** | FAIL → send feedback to Generator → regenerate → re-evaluate |
| **Max Attempts** | Safety valve to prevent infinite loops |

## ⚙️ Step 1: Setup and Azure OpenAI Configuration

Notice we use **two different LLMs** with different temperatures:
- Generator: `temperature=0.7` — allows creative, varied writing.
- Evaluator: `temperature=0.0` — ensures consistent, deterministic grading.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install langchain-openai langchain pydantic

import os
from dotenv import load_dotenv
from typing import List
from pydantic import BaseModel, Field
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Load credentials from .env ─────────────────────────────────────────────────
load_dotenv()

# Generator: creative, varied writing
generator_llm = AzureChatOpenAI(azure_deployment="gpt-4.1", temperature=0.7)

# Evaluator: strict, deterministic grading
evaluator_llm = AzureChatOpenAI(azure_deployment="gpt-4.1", temperature=0.0)

print("Two LLMs initialized:")
print("   Generator: gpt-4.1 (temperature=0.7) — creative writing")
print("   Evaluator: gpt-4.1 (temperature=0.0) — strict grading")

## 📋 Step 2: Defining the Compliance Criteria

We define the exact criteria the memo must meet. These become the Evaluator's grading rubric.

In [ ]:
# Compliance criteria for investment memos
COMPLIANCE_CRITERIA = """
MANDATORY REQUIREMENTS for Investment Memos:
1. INVESTMENT RECOMMENDATION: Must explicitly state Buy, Hold, or Sell.
2. FINANCIAL METRICS: Must mention both Revenue (actual figure) and EPS (actual figure).
3. RISK DISCLAIMER: Must include this exact text: 
   'Investing involves risk. Past performance is not indicative of future results.'
4. PROFESSIONAL TONE: Must be written in third person, professional financial language.
"""

print("✅ Compliance criteria defined:")
print(COMPLIANCE_CRITERIA)

## ✍️ Step 3: Building the Generator

The generator creates the initial draft. When given feedback, it revises the previous draft to address the specific issues.

In [ ]:
generator_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert Financial Analyst specializing in equity research.
    Write professional investment memos for institutional clients.
    
    If you are given FEEDBACK on a previous draft, you MUST revise it to address ALL issues mentioned.
    Output only the memo text — no preamble, no explanation."""),
    ("user", """Request: {request}
    
    {revision_context}""")
])

def run_generator(request: str, feedback: str = None, previous_draft: str = None) -> str:
    """Generate or revise an investment memo."""
    if feedback and previous_draft:
        revision_context = (
            f"PREVIOUS DRAFT (needs revision):\n{previous_draft}\n\n"
            f"FEEDBACK TO ADDRESS:\n{feedback}"
        )
    else:
        revision_context = ""
    
    chain = generator_prompt.partial(revision_context=revision_context) | generator_llm | StrOutputParser()
    return chain.invoke({"request": request})

print("✅ Generator function defined.")

## 🧐 Step 4: Building the Evaluator with Structured Output

The Evaluator uses `with_structured_output()` to force the LLM to return a valid Pydantic object. This is critical — without it, the LLM might return a free-text response that's hard to parse programmatically.

In [ ]:
class EvaluationResult(BaseModel):
    """Structured evaluation result from the compliance officer."""
    decision: str = Field(
        description="Must be exactly 'PASS' or 'FAIL'"
    )
    score: int = Field(
        description="Quality score from 1 (terrible) to 10 (perfect)",
        ge=1, le=10
    )
    missing_requirements: List[str] = Field(
        description="List of mandatory requirements that are missing or incomplete",
        default_factory=list
    )
    feedback: str = Field(
        description="Specific, actionable feedback for the generator to improve the draft. "
                    "If PASS, write 'All requirements met.'"
    )

evaluator_prompt = ChatPromptTemplate.from_messages([
    ("system", f"""You are a strict Compliance Officer and Managing Director at an investment bank.
    
    Review investment memos against these mandatory criteria:
    {COMPLIANCE_CRITERIA}
    
    Be rigorous. If ANY requirement is missing or incomplete, output FAIL.
    Only output PASS if ALL requirements are fully met."""),
    ("user", """Original Request: {{request}}
    
    Memo to Review:
    {{draft}}""")
])

# with_structured_output forces the LLM to return a valid EvaluationResult object
evaluator_chain = evaluator_prompt | evaluator_llm.with_structured_output(EvaluationResult)

print("✅ Evaluator chain defined with structured output.")
print(f"   Output schema: {list(EvaluationResult.model_fields.keys())}")

## 🔄 Step 5: The Evaluator-Optimiser Loop

The main loop runs until the memo passes all criteria or we hit the maximum number of attempts.

In [ ]:
def run_evaluator_optimiser(request: str, max_attempts: int = 4) -> tuple:
    """
    Run the evaluator-optimiser loop.
    
    Args:
        request: The memo writing request
        max_attempts: Maximum number of generation attempts
        
    Returns:
        (final_draft, final_evaluation) tuple
    """
    draft = None
    feedback = None
    history = []
    
    for attempt in range(1, max_attempts + 1):
        print(f"\n{'='*50}")
        print(f"📝 ATTEMPT {attempt}/{max_attempts}")
        print(f"{'='*50}")
        
        # ── Step 1: Generate ──────────────────────────────
        print("✍️  Generator is writing...")
        draft = run_generator(request, feedback, draft)
        print(f"   Draft length: {len(draft)} characters")
        
        # ── Step 2: Evaluate ──────────────────────────────
        print("🧐 Evaluator is reviewing...")
        result = evaluator_chain.invoke({"request": request, "draft": draft})
        
        print(f"   Decision: {result.decision} | Score: {result.score}/10")
        
        history.append({
            "attempt": attempt,
            "decision": result.decision,
            "score": result.score,
            "missing": result.missing_requirements
        })
        
        # ── Step 3: Check Result ──────────────────────────
        if result.decision == "PASS":
            print(f"\n✅ PASSED on attempt {attempt}!")
            return draft, result, history
            
        # If FAIL, prepare feedback for next iteration
        print(f"   Missing: {result.missing_requirements}")
        print(f"   Feedback: {result.feedback[:150]}...")
        feedback = result.feedback
        
    print(f"\n⚠️  Max attempts ({max_attempts}) reached. Returning best effort.")
    return draft, result, history

## 🚀 Step 6: Testing with a Vague Request

Let's deliberately give a vague request that will likely fail the first compliance check. This demonstrates the self-correction loop.

In [ ]:
# Intentionally vague request — likely to miss compliance requirements on first try
request = "Write a quick note about TechCorp's recent earnings. They did pretty well this quarter."

final_draft, final_eval, history = run_evaluator_optimiser(request, max_attempts=4)

print("\n" + "="*60)
print("📊 FINAL APPROVED INVESTMENT MEMO")
print("="*60)
print(final_draft)

## 📈 Step 7: Visualizing the Improvement Journey

In [ ]:
import matplotlib.pyplot as plt

# Plot score progression
attempts = [h["attempt"] for h in history]
scores = [h["score"] for h in history]
decisions = [h["decision"] for h in history]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["green" if d == "PASS" else "red" for d in decisions]
bars = ax.bar(attempts, scores, color=colors, alpha=0.7, edgecolor="black")

# Add score labels on bars
for bar, score, decision in zip(bars, scores, decisions):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f"{score}\n({decision})", ha="center", va="bottom", fontweight="bold")

ax.set_xlabel("Attempt Number", fontsize=12)
ax.set_ylabel("Compliance Score (1-10)", fontsize=12)
ax.set_title("Evaluator-Optimiser: Score Improvement Across Attempts", fontsize=13)
ax.set_ylim(0, 11)
ax.set_xticks(attempts)
ax.axhline(y=8, color="blue", linestyle="--", alpha=0.5, label="Quality threshold (8)")
ax.legend()
plt.tight_layout()
plt.savefig("evaluator_optimiser_scores.png", dpi=100)
plt.show()
print("✅ Score progression chart saved.")

## 📚 Summary and Key Takeaways

| Component | Role | Temperature |
|---|---|---|
| **Generator** | Creates and revises the memo | 0.7 (creative) |
| **Evaluator** | Grades against fixed criteria | 0.0 (strict) |
| **EvaluationResult** | Structured output schema | N/A |
| **Feedback Loop** | FAIL → revise → re-evaluate | N/A |

### 🔑 Key Insights

1. **Separation of Concerns**: The Generator doesn't know the criteria; the Evaluator doesn't write. This specialization improves both creativity and rigor.
2. **Structured Output is Critical**: Using `with_structured_output(Pydantic)` ensures the Evaluator always returns a parseable decision — no regex hacks needed.
3. **Temperature Matters**: The Generator benefits from creativity (0.7); the Evaluator must be deterministic (0.0) to avoid inconsistent grading.
4. **Always set max_attempts**: Prevents infinite loops when the LLM struggles with a particular constraint.

### 🚀 Next Steps
- **Notebook 5**: Learn Parallelisation for running multiple independent analyses simultaneously.
